# Assignment 5 — Option A: "Ask My Resume" RAG Chatbot
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** [Josh Tsutaoka]    
**Option:** A — Resume RAG Chatbot  
**API Path:** [Free]  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Document Loading](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Retrieval Chain](#5-retrieval)
6. [Prompt Engineering](#6-prompting)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the RAG pipeline.  
> See the Option A Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `sentence-transformers` (free path)

In [1]:
%pip install langchain langchain-community chromadb sentence-transformers transformers huggingface_hub python-dotenv pypdf streamlit pandas numpy
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

for f in Path(".").iterdir():
    print(f.name)

.env
.ipynb_checkpoints
A5_OptionA_Resume_RAG_Starter.ipynb
cover_letter.pdf
Linkedin_about.txt
rag_pipeline.ipynb
requirements.txt
Resume.pdf


In [4]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

print("Hugging Face token loaded:", hf_token is not None)

Hugging Face token loaded: True


---
<a id="2-loading"></a>
## 2. Document Loading

Load your 3-5 career documents from `data/`.  
Print: number of documents loaded, document types, and a sample of content to verify.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [7]:
# ── Load your documents ──
DATA_DIR = Path(".")
documents = []

for file_path in DATA_DIR.iterdir():
    if file_path.suffix.lower() == ".pdf":
        loader = PyPDFLoader(str(file_path))
        documents.extend(loader.load())
    elif file_path.suffix.lower() == ".txt" and file_path.name.lower() not in ["requirements.txt", "env.txt"]:
        loader = TextLoader(str(file_path), encoding="utf-8")
        documents.extend(loader.load())

print(f"Loaded {len(documents)} document pages/sections.")

Loaded 3 document pages/sections.


In [8]:
for i, doc in enumerate(documents):
    print(f"\n--- Document {i+1} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:500])


--- Document 1 ---
Source: cover_letter.pdf
Hiring  Committee  
Divergent
 
Torrance,
 
CA
 
Dear  Hiring  Committee,  
I  am  writing  to  express  my  interest  in  the  Data  and  Process  Analytics  Intern  position  at  
Divergent
 
for
 
Summer
 
2026.
 
My
 
hands-on
 
experience
 
with
 
Python
 
data
 
processing,
 
ETL
 
pipeline
 
development,
 
dashboard
 
building,
 
and
 
analytical
 
modeling,
 
combined
 
with
 
my
 
academic
 
work
 
in
 
Business
 
Analytics,
 
makes
 
me
 
enthusiastic
 
about
 
contributing
 
to
 
Dive

--- Document 2 ---
Source: Linkedin_about.txt
I'm an MSBA candidate at Loyola Marymount University graduating August 2026, with a background in Economics from Gonzaga and hands-on experience turning messy, multi-source data into structured outputs that help people make better decisions.
 
My work spans donor analytics at a data consultancy, program effectiveness analysis at a nonprofit, and academic research involving large-scale web scraping and NL

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively (chunk count, avg length), and justify your final choice.

In [11]:
# Strategy 1: fixed-size chunking
fixed_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

fixed_chunks = fixed_splitter.split_documents(documents)

# Summary statistics for fixed-size chunks
fixed_lengths = [len(chunk.page_content) for chunk in fixed_chunks]

print("Fixed-size chunking")
print(f"Number of chunks: {len(fixed_chunks)}")
print(f"Average chunk length: {np.mean(fixed_lengths):.2f}")
print(f"Minimum chunk length: {np.min(fixed_lengths)}")
print(f"Maximum chunk length: {np.max(fixed_lengths)}")

for i, chunk in enumerate(fixed_chunks[:3], start=1):
    print(f"\n--- Fixed Chunk {i} ---")
    print("Source:", chunk.metadata.get("source"))
    print(chunk.page_content[:500])

Created a chunk of size 892, which is longer than the specified 500


Fixed-size chunking
Number of chunks: 4
Average chunk length: 2705.25
Minimum chunk length: 67
Maximum chunk length: 6515

--- Fixed Chunk 1 ---
Source: cover_letter.pdf
Hiring  Committee  
Divergent
 
Torrance,
 
CA
 
Dear  Hiring  Committee,  
I  am  writing  to  express  my  interest  in  the  Data  and  Process  Analytics  Intern  position  at  
Divergent
 
for
 
Summer
 
2026.
 
My
 
hands-on
 
experience
 
with
 
Python
 
data
 
processing,
 
ETL
 
pipeline
 
development,
 
dashboard
 
building,
 
and
 
analytical
 
modeling,
 
combined
 
with
 
my
 
academic
 
work
 
in
 
Business
 
Analytics,
 
makes
 
me
 
enthusiastic
 
about
 
contributing
 
to
 
Dive

--- Fixed Chunk 2 ---
Source: Linkedin_about.txt
I'm an MSBA candidate at Loyola Marymount University graduating August 2026, with a background in Economics from Gonzaga and hands-on experience turning messy, multi-source data into structured outputs that help people make better decisions.
 
My work spans donor analytics at a 

In [12]:
# ── Strategy 2 ──
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

recursive_chunks = recursive_splitter.split_documents(documents)

# Summary statistics for recursive chunks
recursive_lengths = [len(chunk.page_content) for chunk in recursive_chunks]

print("Recursive chunking")
print(f"Number of chunks: {len(recursive_chunks)}")
print(f"Average chunk length: {np.mean(recursive_lengths):.2f}")
print(f"Minimum chunk length: {np.min(recursive_lengths)}")
print(f"Maximum chunk length: {np.max(recursive_lengths)}")

for i, chunk in enumerate(recursive_chunks[:3], start=1):
    print(f"\n--- Recursive Chunk {i} ---")
    print("Source:", chunk.metadata.get("source"))
    print(chunk.page_content[:500])

Recursive chunking
Number of chunks: 31
Average chunk length: 369.48
Minimum chunk length: 67
Maximum chunk length: 496

--- Recursive Chunk 1 ---
Source: cover_letter.pdf
Hiring  Committee  
Divergent
 
Torrance,
 
CA
 
Dear  Hiring  Committee,  
I  am  writing  to  express  my  interest  in  the  Data  and  Process  Analytics  Intern  position  at  
Divergent
 
for
 
Summer
 
2026.
 
My
 
hands-on
 
experience
 
with
 
Python
 
data
 
processing,
 
ETL
 
pipeline
 
development,
 
dashboard
 
building,
 
and
 
analytical
 
modeling,
 
combined
 
with
 
my
 
academic
 
work
 
in
 
Business
 
Analytics,
 
makes
 
me
 
enthusiastic
 
about
 
contributing
 
to

--- Recursive Chunk 2 ---
Source: cover_letter.pdf
me
 
enthusiastic
 
about
 
contributing
 
to
 
Divergent
 
and
 
the
 
development
 
of
 
the
 
Divergent
 
Adaptive
 
Production
 
System.
 
During  my  Data  Analytics  Internship  at  Guardian  Insight  Group,  I  built  and  maintained  Python  
and
 
MySQL
 
ETL
 
pipelines
 

In [13]:
# ── Compare strategies ──

comparison_data = {
    "Strategy": ["Fixed-size chunking", "Recursive chunking"],
    "Number of Chunks": [len(fixed_chunks), len(recursive_chunks)],
    "Average Chunk Length": [
        np.mean([len(chunk.page_content) for chunk in fixed_chunks]),
        np.mean([len(chunk.page_content) for chunk in recursive_chunks])
    ],
    "Minimum Chunk Length": [
        np.min([len(chunk.page_content) for chunk in fixed_chunks]),
        np.min([len(chunk.page_content) for chunk in recursive_chunks])
    ],
    "Maximum Chunk Length": [
        np.max([len(chunk.page_content) for chunk in fixed_chunks]),
        np.max([len(chunk.page_content) for chunk in recursive_chunks])
    ]
}

chunk_comparison_df = pd.DataFrame(comparison_data)

chunk_comparison_df

print("FIXED-SIZE CHUNK EXAMPLE")
print("------------------------")
print("Source:", fixed_chunks[0].metadata.get("source"))
print(fixed_chunks[0].page_content[:700])

print("\n\nRECURSIVE CHUNK EXAMPLE")
print("------------------------")
print("Source:", recursive_chunks[0].metadata.get("source"))
print(recursive_chunks[0].page_content[:700])

FIXED-SIZE CHUNK EXAMPLE
------------------------
Source: cover_letter.pdf
Hiring  Committee  
Divergent
 
Torrance,
 
CA
 
Dear  Hiring  Committee,  
I  am  writing  to  express  my  interest  in  the  Data  and  Process  Analytics  Intern  position  at  
Divergent
 
for
 
Summer
 
2026.
 
My
 
hands-on
 
experience
 
with
 
Python
 
data
 
processing,
 
ETL
 
pipeline
 
development,
 
dashboard
 
building,
 
and
 
analytical
 
modeling,
 
combined
 
with
 
my
 
academic
 
work
 
in
 
Business
 
Analytics,
 
makes
 
me
 
enthusiastic
 
about
 
contributing
 
to
 
Divergent
 
and
 
the
 
development
 
of
 
the
 
Divergent
 
Adaptive
 
Production
 
System.
 
During  my  Data  Analytics  Internship  at  Guardian  Insight  Group,  I  built  and  maintained  Python  
a


RECURSIVE CHUNK EXAMPLE
------------------------
Source: cover_letter.pdf
Hiring  Committee  
Divergent
 
Torrance,
 
CA
 
Dear  Hiring  Committee,  
I  am  writing  to  express  my  interest  in  the  Data  and  Process  

In [14]:
final_chunks = recursive_chunks

print("Final chunking strategy selected: RecursiveCharacterTextSplitter")
print(f"Final number of chunks: {len(final_chunks)}")

Final chunking strategy selected: RecursiveCharacterTextSplitter
Final number of chunks: 31


### Chunking Decision

**Which strategy did you choose?**  
**Why?**  
**Final settings (chunk_size, overlap):**

I chose recursive chunking because the documents contain short sections, bullet points, project descriptions, and experience summaries. A basic fixed-size splitter can break these sections in awkward places, which may separate a skill from the project or experience that explains it. Recursive chunking is better for this use case because it tries to preserve larger text boundaries first, such as paragraphs and line breaks, before splitting into smaller pieces.

The final settings I used were:

- `chunk_size = 500`
- `chunk_overlap = 50`
- `separators = ["\n\n", "\n", " ", ""]`
I chose a chunk size of 500 because it is large enough to capture meaningful context from a resume or project description, but small enough to keep retrieval focused. I used an overlap of 50 characters so that important information near the edge of one chunk is not completely lost in the next chunk. The separator order helps the splitter prioritize cleaner breaks, starting with paragraph breaks, then line breaks, then spaces, and finally individual characters if needed.

Overall, recursive chunking was the better choice because it created more readable chunks and should improve the chatbot’s ability to retrieve complete, relevant evidence when answering recruiter-style questions.

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [15]:
# ── Create embeddings and vector store ──

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Free embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create ChromaDB vector store from final chunks
vectordb = Chroma.from_documents(
    documents=final_chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

print("Vector store created successfully.")
print(f"Number of chunks stored: {len(final_chunks)}")

C:\Users\joshk\AppData\Local\Temp\ipykernel_47484\1952839831.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\joshk\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joshk\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully.
Number of chunks stored: 31


In [16]:
# ── Verify: run a test similarity search ──
test_query = "Python data analytics and machine learning experience"

results = vectordb.similarity_search(test_query, k=3)

print(f"Search query: {test_query}")
print(f"Number of results returned: {len(results)}")

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])

Search query: Python data analytics and machine learning experience
Number of results returned: 3

--- Result 1 ---
Source: Resume.pdf
SKILLS
 •  Analytics  &  Programming :  Python  (pandas,  NumPy,  scikit-learn),  SQL,  R,  Excel  —  with  applied  experience  in  regression,  
classification,
 
clustering,
 
and
 
EDA
 •  Business  Intelligence  &  Visualization :  Tableau,  Dashboard  Development,  KPI  Tracking,  Data  Storytelling,  Executive  Reporting,  
Cross-Functional
 
Communication

--- Result 2 ---
Source: cover_letter.pdf
me
 
enthusiastic
 
about
 
contributing
 
to
 
Divergent
 
and
 
the
 
development
 
of
 
the
 
Divergent
 
Adaptive
 
Production
 
System.
 
During  my  Data  Analytics  Internship  at  Guardian  Insight  Group,  I  built  and  maintained  Python  
and
 
MySQL
 
ETL
 
pipelines
 
to
 
clean,
 
structure,
 
and
 
transform
 
large
 
datasets.
 
I
 
also
 
created
 
dashboards
 
and
 
reporting
 
tools
 
to
 
support
 
operational
 
decision
 
making.


---
<a id="5-retrieval"></a>
## 5. Retrieval Chain

Connect your vector store to an LLM to build the full RAG chain.

**Paid path:** OpenAI `gpt-4o-mini` or `gpt-3.5-turbo`  
**Free path:** HuggingFace Inference API or Ollama

In [30]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

print("OpenAI API key loaded:", openai_api_key is not None and openai_api_key.startswith("sk-"))

OpenAI API key loaded: True


In [33]:
%pip install langchain-openai openai

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 17.3 MB/s  0:00:00
   ---------------------------------------- 0.0/879.1 kB ? eta -:--:--
   ---------------------------------------- 879.1/879.1 kB 38.1 MB/s  0:00:00

   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   --------------------


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

print("OpenAI API key loaded:", openai_api_key is not None and openai_api_key.startswith("sk-"))

OpenAI API key loaded: True


In [36]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    api_key=openai_api_key
)

print("OpenAI LLM initialized successfully.")

OpenAI LLM initialized successfully.


In [38]:
test_response = llm.invoke("Say hello in one sentence.")
print(test_response.content)

Hello! How can I assist you today?


In [19]:
from langchain_huggingface import HuggingFaceEndpoint
# ── Initialize LLM ──

llm = HuggingFaceEndpoint(
    repo_id="google/flan-t5-base",
    task="text2text-generation",
    max_new_tokens=300,
    temperature=0.2,
    huggingfacehub_api_token=hf_token
)

print("Hugging Face LLM initialized successfully.")

Hugging Face LLM initialized successfully.


In [22]:
%pip install transformers torch langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
# ── Build retrieval chain ──
def build_prompt(question, retrieved_docs):
    """
    Builds a grounded prompt using the user's question and retrieved document chunks.
    """
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    prompt = f"""
You are a professional career assistant answering questions about Josh Tsutaoka's background.

Use only the provided context to answer the question.
Do not make up information.
If the answer is not available in the context, say:
"I do not have enough information in the provided documents to answer that."

Keep the answer concise, professional, and recruiter-friendly.

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt


def ask_resume_bot(question, k=3):
    """
    Manual RAG chain:
    1. Retrieve relevant chunks from ChromaDB
    2. Build a grounded prompt
    3. Send the prompt to OpenAI gpt-4o-mini
    4. Return the answer and retrieved sources
    """
    retrieved_docs = vectordb.similarity_search(question, k=k)
    prompt = build_prompt(question, retrieved_docs)
    response = llm.invoke(prompt)
    
    return {
        "question": question,
        "answer": response.content,
        "source_documents": retrieved_docs
    }

print("Manual retrieval chain updated successfully.")

Manual retrieval chain updated successfully.


In [43]:
test_question = "What technical skills does Josh have?"

response = ask_resume_bot(test_question, k=3)

print("Question:")
print(response["question"])

print("\nAnswer:")
print(response["answer"])

print("\nRetrieved Sources:")
for i, doc in enumerate(response["source_documents"], start=1):
    print(f"\n--- Source {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:500])

Question:
What technical skills does Josh have?

Answer:
Josh has technical skills in analytics and programming, including Python (with libraries such as pandas, NumPy, and scikit-learn), SQL, R, and Excel. He has applied experience in regression, classification, clustering, and exploratory data analysis (EDA). Additionally, he is skilled in business intelligence and visualization tools such as Tableau, dashboard development, KPI tracking, data storytelling, executive reporting, and cross-functional communication.

Retrieved Sources:

--- Source 1 ---
Source: Resume.pdf
JOSH  TSUTAOKA
 
+1  (650)  504-9576   |   joshkt2003@gmail.com   |   linkedin.com/in/josh-tsutaoka   |   https://github.com/jtsu03 
EDUCATION

--- Source 2 ---
Source: cover_letter.pdf
making.
 
This
 
work
 
strengthened
 
my
 
ability
 
to
 
take
 
raw
 
data
 
and
 
convert
 
it
 
into
 
clear
 
and
 
actionable
 
insights.
 
These
 
skills
 
align
 
directly
 
with
 
the
 
tasks
 
involved
 
in
 
capturing
 
and
 


---
<a id="6-prompting"></a>
## 6. Prompt Engineering

Design your system prompt for the RAG chain. Consider: grounding (answer only from context), tone, how to handle out-of-scope questions, response format.

**Required:** Show at least 3 iterations. For each, explain what you changed, why, and show a before/after test with the same question.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [44]:
# ── Your prompt iterations and testing below ──
# Add as many cells as you need
prompt_v1 = "Answer questions about my professional background. Use the Context. Answer the question clearly. What technical skills does Josh have?"

In [45]:
prompt_test_question = "What technical skills does Josh have?"

In [46]:
def build_prompt_v1(question, retrieved_docs):
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    full_prompt = f"""
{prompt_v1}

Context:
{context}

Question:
{question}

Answer:
"""
    return full_prompt

In [47]:
def ask_resume_bot_v1(question, k=3):
    retrieved_docs = vectordb.similarity_search(question, k=k)
    prompt = build_prompt_v1(question, retrieved_docs)
    response = llm.invoke(prompt)
    
    return {
        "question": question,
        "answer": response.content,
        "source_documents": retrieved_docs
    }

In [48]:
response_v1 = ask_resume_bot_v1(prompt_test_question, k=3)

print("Prompt Version 1 Answer:")
print(response_v1["answer"])

print("\nRetrieved Sources:")
for i, doc in enumerate(response_v1["source_documents"], start=1):
    print(f"\n--- Source {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:400])

Prompt Version 1 Answer:
Josh has the following technical skills:

1. **Analytics & Programming**: Proficient in Python (including libraries such as pandas, NumPy, and scikit-learn), SQL, R, and Excel. He has applied experience in regression, classification, clustering, and exploratory data analysis (EDA).

2. **Business Intelligence & Visualization**: Skilled in Tableau, dashboard development, KPI tracking, data storytelling, executive reporting, and cross-functional communication.

Retrieved Sources:

--- Source 1 ---
Source: Resume.pdf
JOSH  TSUTAOKA
 
+1  (650)  504-9576   |   joshkt2003@gmail.com   |   linkedin.com/in/josh-tsutaoka   |   https://github.com/jtsu03 
EDUCATION

--- Source 2 ---
Source: cover_letter.pdf
making.
 
This
 
work
 
strengthened
 
my
 
ability
 
to
 
take
 
raw
 
data
 
and
 
convert
 
it
 
into
 
clear
 
and
 
actionable
 
insights.
 
These
 
skills
 
align
 
directly
 
with
 
the
 
tasks
 
involved
 
in
 
capturing
 
and
 
organizing
 
factory
 
data
 
fo

### Prompt Version 1: Basic Resume Assistant

For the first prompt, I started with a simple instruction that told the model to answer questions about my professional background using the provided context. This version was intentionally basic so I could establish a baseline before adding stronger rules.

**Prompt focus:**
- Answer questions about my background
- Use the retrieved context
- Respond clearly and directly

**Test question used:**
`What technical skills does Josh have?`

**Result:**
Prompt Version 1 was able to answer the question and identify relevant technical skills from the retrieved documents, including programming languages, analytics tools, and visualization experience. However, the prompt did not strongly tell the model to avoid outside knowledge or unsupported assumptions.

**What I learned:**
This version worked for a straightforward factual question, but it needed stronger grounding instructions. Since this is a RAG chatbot, the answer should be based only on the retrieved resume, cover letter, and LinkedIn context. This led to Prompt Version 2, where I added clearer instructions not to guess or use outside knowledge.

**Before improvement:**  
Prompt Version 1 gave a clear answer, but the instructions were general.

**Improvement needed:**  
The next version should include stronger grounding language so the chatbot only answers from the retrieved documents and does not make unsupported assumptions.

In [49]:
prompt_v2 = """
Answer questions about my professional background using only the provided context.
Do not use outside knowledge or make assumptions.
If the context does not contain enough information to answer the question, say that the information is not available in the provided documents.
"""

In [50]:
def build_prompt_v2(question, retrieved_docs):
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    full_prompt = f"""
{prompt_v2}

Context:
{context}

Question:
{question}

Answer:
"""
    return full_prompt

In [51]:
def ask_resume_bot_v2(question, k=3):
    retrieved_docs = vectordb.similarity_search(question, k=k)
    prompt = build_prompt_v2(question, retrieved_docs)
    response = llm.invoke(prompt)
    
    return {
        "question": question,
        "answer": response.content,
        "source_documents": retrieved_docs
    }

In [52]:
response_v2 = ask_resume_bot_v2(prompt_test_question, k=3)

print("Prompt Version 2 Answer:")
print(response_v2["answer"])

print("\nRetrieved Sources:")
for i, doc in enumerate(response_v2["source_documents"], start=1):
    print(f"\n--- Source {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:400])

Prompt Version 2 Answer:
Josh has technical skills in the following areas:

- Analytics & Programming: Python (with libraries such as pandas, NumPy, scikit-learn), SQL, R, and Excel, with applied experience in regression, classification, clustering, and exploratory data analysis (EDA).
- Business Intelligence & Visualization: Tableau, dashboard development, KPI tracking, data storytelling, executive reporting, and cross-functional communication.

Retrieved Sources:

--- Source 1 ---
Source: Resume.pdf
JOSH  TSUTAOKA
 
+1  (650)  504-9576   |   joshkt2003@gmail.com   |   linkedin.com/in/josh-tsutaoka   |   https://github.com/jtsu03 
EDUCATION

--- Source 2 ---
Source: cover_letter.pdf
making.
 
This
 
work
 
strengthened
 
my
 
ability
 
to
 
take
 
raw
 
data
 
and
 
convert
 
it
 
into
 
clear
 
and
 
actionable
 
insights.
 
These
 
skills
 
align
 
directly
 
with
 
the
 
tasks
 
involved
 
in
 
capturing
 
and
 
organizing
 
factory
 
data
 
for
 
simulation
 
and
 
decision
 
supp

### Prompt Version 2: Grounded Context-Only Assistant

For the second prompt, I revised the original prompt by adding stronger grounding instructions. Instead of only telling the model to use the provided context, I specifically instructed it to answer **only** from the context and avoid using outside knowledge or assumptions.

**What I changed from Version 1:**
- Added a rule to use only the provided context
- Added a rule not to use outside knowledge
- Added a rule not to make assumptions
- Added instructions for what to do if the answer is missing from the documents

**Why I changed it:**
Prompt Version 1 answered the test question clearly, but it did not strongly prevent the model from adding information that may not be in the retrieved documents. Since this project is a RAG chatbot, the response needs to be grounded in the resume, cover letter, and LinkedIn About section rather than the model’s general knowledge.

**Test question used:**
`What technical skills does Josh have?`

**Result:**
Prompt Version 2 produced a more grounded answer because the model was directly instructed to rely only on the retrieved context. The response still identified Josh’s technical skills, but the prompt made the answer more appropriate for a RAG system because it reduced the risk of unsupported claims.

**What I learned:**
Adding grounding instructions improved the reliability of the chatbot. However, the response format was still fairly general. For the final prompt, I wanted to improve the tone and formatting so the answer would sound more professional and recruiter-friendly.

**Before improvement:**  
Prompt Version 1 gave a clear answer, but it did not strongly restrict the model to the retrieved documents.

**After improvement:**  
Prompt Version 2 added grounding rules so the chatbot would answer only from the provided context and avoid making assumptions.

**Improvement needed for Version 3:**  
The next version should keep the grounding rules but improve the response style by adding a professional tone, concise formatting, and clearer handling for out-of-scope questions.

In [53]:
prompt_v3 = """
You are a professional career assistant answering questions about my background.

Use only the provided context from my resume, cover letter, and LinkedIn About section.
Do not use outside knowledge, guess, or add information that is not supported by the context.

If the context does not contain enough information to answer the question, respond:
"The provided documents do not include enough information to answer that."

Keep the response concise, professional, and recruiter-friendly.
Use bullet points when they make the answer easier to read.
For job-fit questions, explain the answer using specific evidence from the provided context.
"""

In [54]:
def build_prompt_v3(question, retrieved_docs):
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    full_prompt = f"""
{prompt_v3}

Context:
{context}

Question:
{question}

Answer:
"""
    return full_prompt

In [55]:
def ask_resume_bot_v3(question, k=3):
    retrieved_docs = vectordb.similarity_search(question, k=k)
    prompt = build_prompt_v3(question, retrieved_docs)
    response = llm.invoke(prompt)
    
    return {
        "question": question,
        "answer": response.content,
        "source_documents": retrieved_docs
    }

In [56]:
response_v3 = ask_resume_bot_v3(prompt_test_question, k=3)

print("Prompt Version 3 Answer:")
print(response_v3["answer"])

print("\nRetrieved Sources:")
for i, doc in enumerate(response_v3["source_documents"], start=1):
    print(f"\n--- Source {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:400])

Prompt Version 3 Answer:
Josh has the following technical skills:

- **Analytics & Programming:**
  - Python (pandas, NumPy, scikit-learn)
  - SQL
  - R
  - Excel
  - Applied experience in regression, classification, clustering, and exploratory data analysis (EDA)

- **Business Intelligence & Visualization:**
  - Tableau
  - Dashboard Development
  - KPI Tracking
  - Data Storytelling
  - Executive Reporting
  - Cross-Functional Communication

Retrieved Sources:

--- Source 1 ---
Source: Resume.pdf
JOSH  TSUTAOKA
 
+1  (650)  504-9576   |   joshkt2003@gmail.com   |   linkedin.com/in/josh-tsutaoka   |   https://github.com/jtsu03 
EDUCATION

--- Source 2 ---
Source: cover_letter.pdf
making.
 
This
 
work
 
strengthened
 
my
 
ability
 
to
 
take
 
raw
 
data
 
and
 
convert
 
it
 
into
 
clear
 
and
 
actionable
 
insights.
 
These
 
skills
 
align
 
directly
 
with
 
the
 
tasks
 
involved
 
in
 
capturing
 
and
 
organizing
 
factory
 
data
 
for
 
simulation
 
and
 
decision
 
support

### Prompt Version 3: Final Recruiter-Friendly Grounded Assistant

For the third prompt, I revised Prompt Version 2 by keeping the grounding rules and adding clearer instructions for tone, formatting, and job-fit questions. This version was designed to make the chatbot sound more professional and useful for a recruiter or hiring manager.

**What I changed from Version 2:**
- Kept the rule to answer only from the provided context
- Kept the rule not to guess or use outside knowledge
- Added a more professional and recruiter-friendly tone
- Added instructions to keep answers concise
- Added instructions to use bullet points when helpful
- Added instructions for job-fit questions, requiring the answer to use evidence from the retrieved documents
- Made the out-of-scope response clearer

**Why I changed it:**
Prompt Version 2 improved grounding, but the response style could still be improved. Since the goal of this project is to build a chatbot that could answer recruiter-style questions, the final prompt needed to produce answers that were not only accurate, but also clear, polished, and easy to read.

**Test question used:**
`What technical skills does Josh have?`

**Result:**
Prompt Version 3 produced the strongest response because it kept the answer grounded in the retrieved documents while also improving the structure and tone. The answer was more recruiter-friendly and easier to scan because the prompt encouraged concise formatting and evidence-based responses.

**What I learned:**
The final prompt worked best because it balanced accuracy, grounding, and communication style. For a RAG chatbot, it is not enough for the model to answer correctly. It also needs to avoid unsupported claims, handle missing information appropriately, and present the answer in a professional format.

**Before improvement:**  
Prompt Version 2 improved grounding by requiring the chatbot to answer only from the provided context, but it did not give as much guidance on tone, formatting, or job-fit questions.

**After improvement:**  
Prompt Version 3 kept the grounding rules and added recruiter-friendly formatting instructions. This made the chatbot better suited for professional use because answers were clearer, more concise, and easier to evaluate.

**Final prompt selected:**  
Prompt Version 3 was selected as the final system prompt because it provided the best balance of grounded answers, professional tone, clear formatting, and appropriate handling of missing information.

---
<a id="7-evaluation"></a>
## 7. Evaluation

Test your chatbot with **10 questions** across 4 categories:
- Factual retrieval (2-3): questions with clear answers in your docs
- Inference (2-3): questions requiring reasoning across your docs
- Out-of-scope (2-3): questions your docs cannot answer
- Specificity (2-3): questions targeting a specific document

For each question, score: **retrieval quality** (Yes/Partial/No), **faithfulness** (Faithful/Partial/Hallucinated), **answer quality** (1-5).

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [57]:
# ── Define your 10 test questions and run evaluation ──
evaluation_questions = [
    {
        "id": 1,
        "category": "Factual retrieval",
        "question": "What technical skills does Josh have?"
    },
    {
        "id": 2,
        "category": "Factual retrieval",
        "question": "What programming languages and tools does Josh mention?"
    },
    {
        "id": 3,
        "category": "Factual retrieval",
        "question": "What is Josh currently studying?"
    },
    {
        "id": 4,
        "category": "Inference",
        "question": "Would Josh be a good fit for a data analyst role?"
    },
    {
        "id": 5,
        "category": "Inference",
        "question": "What are Josh's strongest transferable skills?"
    },
    {
        "id": 6,
        "category": "Inference",
        "question": "Would Josh be a good fit for a process analytics internship?"
    },
    {
        "id": 7,
        "category": "Out-of-scope",
        "question": "What is Josh's salary expectation?"
    },
    {
        "id": 8,
        "category": "Out-of-scope",
        "question": "What is Josh's GPA?"
    },
    {
        "id": 9,
        "category": "Specificity",
        "question": "What does Josh's cover letter say about his ETL experience?"
    },
    {
        "id": 10,
        "category": "Specificity",
        "question": "What does Josh's LinkedIn About section say about his background?"
    }
]

In [58]:
evaluation_results = []

for item in evaluation_questions:
    response = ask_resume_bot_v3(item["question"], k=3)
    
    evaluation_results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "answer": response["answer"],
        "source_documents": response["source_documents"]
    })

print("Evaluation completed.")
print(f"Number of questions tested: {len(evaluation_results)}")

Evaluation completed.
Number of questions tested: 10


In [61]:
print("\nRetrieved Sources:")
for i, doc in enumerate(result["source_documents"], start=1):
    print(f"\n--- Source {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:500])


Retrieved Sources:

--- Source 1 ---
Source: Resume.pdf
JOSH  TSUTAOKA
 
+1  (650)  504-9576   |   joshkt2003@gmail.com   |   linkedin.com/in/josh-tsutaoka   |   https://github.com/jtsu03 
EDUCATION

--- Source 2 ---
Source: Linkedin_about.txt
Portfolio: github.com/jtsu03/Josh_Tsutaoka_data_analytics_portfolio

--- Source 3 ---
Source: Resume.pdf
actionable
 
insights
 
for
 
leadership
 •  Translated  analytical  findings  into  multiple  deliverable  formats  —  dashboards,  website  content,  newsletter  visuals,  brochure  
graphics,
 
and
 
a
 
founder
 
presentation
 
—
 
used
 
to
 
support
 
government
 
funding
 
applications
 
and
 
communicate
 
program
 
impact
 
to
 
external
 
stakeholders


In [62]:
for result in evaluation_results:
    print("=" * 100)
    print(f"Question {result['id']}: {result['question']}")
    print(f"Category: {result['category']}")

    print("\nAnswer:")
    print(result["answer"])

    print("\n")

Question 1: What technical skills does Josh have?
Category: Factual retrieval

Answer:
Josh has the following technical skills:

- **Analytics & Programming:**
  - Python (pandas, NumPy, scikit-learn)
  - SQL
  - R
  - Excel
  - Applied experience in regression, classification, clustering, and exploratory data analysis (EDA)

- **Business Intelligence & Visualization:**
  - Tableau
  - Dashboard Development
  - KPI Tracking
  - Data Storytelling
  - Executive Reporting
  - Cross-Functional Communication


Question 2: What programming languages and tools does Josh mention?
Category: Factual retrieval

Answer:
Josh mentions the following programming languages and tools:

- **Programming Languages:**
  - Python (with libraries: pandas, NumPy, scikit-learn)
  - SQL
  - R
  - Excel

- **Tools & Platforms:**
  - Jupyter Notebook
  - MySQL Workbench
  - DBeaver
  - GitHub
  - Microsoft 365
  - Tableau


Question 3: What is Josh currently studying?
Category: Factual retrieval

Answer:
Josh is 

## Evaluation Results Summary

The chatbot was tested using 10 questions across four categories: factual retrieval, inference, out-of-scope, and specificity. Overall, the chatbot performed well on factual retrieval and inference questions, but it struggled with some specificity questions that targeted a particular document.

### Evaluation Table

| # | Category | Question | Retrieval Quality | Answer Faithfulness | Answer Quality |
|---|---|---|---|---|---|
| 1 | Factual retrieval | What technical skills does Josh have? | Yes | Faithful | 5 |
| 2 | Factual retrieval | What programming languages and tools does Josh mention? | Yes | Faithful | 5 |
| 3 | Factual retrieval | What is Josh currently studying? | Yes | Faithful | 5 |
| 4 | Inference | Would Josh be a good fit for a data analyst role? | Yes | Faithful | 5 |
| 5 | Inference | What are Josh's strongest transferable skills? | Partial | Partial | 4 |
| 6 | Inference | Would Josh be a good fit for a process analytics internship? | Yes | Faithful | 5 |
| 7 | Out-of-scope | What is Josh's salary expectation? | Yes | Faithful | 5 |
| 8 | Out-of-scope | What is Josh's GPA? | Yes | Faithful | 5 |
| 9 | Specificity | What does Josh's cover letter say about his ETL experience? | No | Faithful | 2 |
| 10 | Specificity | What does Josh's LinkedIn About section say about his background? | No | Faithful | 2 |


### Evaluation Analysis

**Where does the chatbot succeed?**
It succeeds most when the question is broad and the answer is clearly available in the documents. It performed well on factual questions about skills, languages, tools and education. It also did well on braoder inference questions. It handled out-of-scope questions well too.

**Where does it fail? Why?**
The chatbot fials on document-specific questions. It struggled when asked about the cover letter and Linked section because it said the documents did not contain enough information, even if the content existed. This happened because the retriever only pulled the top 3 chunks and did not always grab the document specific section. Increasing k or adding filterting could improve it.

**What would you improve?**
I would improve the retrieval step, the chatbot worked well with broad questions but missed specific answers. I would also add more documents, it is hard to completely learn when there is only 3-5 documents being provided as data. I would also add some filtering so if a question mentions cover letter or Linkedin the retriever would prioritize those documents.

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option A*